<a href="https://colab.research.google.com/github/AI4Finance-Foundation/ElegantRL/blob/master/tutorial_BipedalWalker_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **BipedalWalker-v3 Example in Stable-Baselines3**






![biped_walker_url](https://cdn.rawgit.com/hardmaru/pybullet_animations/f6f7fcd7/anim/bipedstoc/biped_noisy.gif)


# **Task Description**

[BipedalWalker-v3](https://gym.openai.com/envs/BipedalWalker-v2/) is one of the most challenging continuous control environments in OpenAI Gym, requiring an agent to master bipedal locomotion - a fundamental skill in robotics! In this task, we aim to train a 2D bipedal robot to walk efficiently through rough terrain.

What makes this environment particularly interesting is its combination of challenges:
- Continuous action space (joint torques must be precisely controlled)
- High-dimensional state space (24 observations including joint angles, velocities, and terrain information)
- Balance requirements (the robot must maintain stability while moving)
- Terrain variability (especially in hardcore mode with obstacles)

Successfully training an agent for this task requires sophisticated reinforcement learning techniques, as only a few RL implementations can consistently achieve the target reward of 300+ points. The environment serves as an excellent benchmark for testing RL algorithms and their ability to learn complex motor control policies.

To monitor the experiment, create an account in W&B and copy to your clipboard the API key ([link](https://wandb.ai/site)).


# **Part 1: Install Stable-Baselines3**
 [Stable Baselines3 (SB3)](https://stable-baselines3.readthedocs.io/en/master/) is a set of reliable implementations of reinforcement learning algorithms in PyTorch. [RL Baselines3 Zoo](https://github.com/DLR-RM/rl-baselines3-zoo) provides a collection of pre-trained agents, scripts for training, evaluating agents, tuning hyperparameters, plotting results and recording videos.

In [ ]:
# install required libraries
!pip install swig       # Wrapper to avoid conflicts with wheel
!pip install 'stable-baselines3[extra]' # RL algorithms
!pip install wandb      # Important for Debugging and Data Logging
!pip install box2d-py   # To define a continuous action space (joint angles-4)

# **Part 2: Import Packages**



In [ ]:
import gymnasium as gym # Standard API for reinforcement learning, and a diverse collection of reference environments
import os
import wandb
from gym.wrappers import GrayScaleObservation
from wandb.integration.sb3 import WandbCallback
from stable_baselines3 import PPO  # Proximal Policy Optimization
from stable_baselines3.common.vec_env import DummyVecEnv,VecVideoRecorder, VecFrameStack # Record Video
from stable_baselines3.common.evaluation import evaluate_policy # Evaluation of the Policy in Training
from stable_baselines3.common.monitor import Monitor # Oversees the experiment

# **Part 3: Check connection to Google Compute Engine Backend**
In the top-right corner of this window, you will see a green checkmark and, next to it, the runtime type. Since we want to use a GPU instead of a CPU, we need to ensure that the runtime type is set to "GPU." If you want to accelerate your machine learning workloads on Google Colab, you can choose between GPU or TPU runtimes.

While Tensor Processing Units (TPUs) are Google's custom-designed application-specific integrated circuits (ASICs) created to accelerate machine learning workloads, GPUs are more commonly used for general deep learning tasks.
Using GPUs and TPUs on Google Colab is free, though with some usage limitations. If the correct runtime type is not already selected, you can change it by clicking the dropdown menu and selecting "Change runtime type." Choose "GPU" from the runtime type options. Without this change, the training process will take an exceptionally long time!



In [ ]:
print("These are the current specs of the GPU!")
!nvidia-smi

These are the current specs of the GPU!
Fri Nov  7 01:10:03 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P0             28W /   70W |     104MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------

# **Part 4: Algorithm**

## **Part A: Create the environment**
An environment is a conceptual or computational representation of the system or world in which an agent interacts.

In [ ]:
def make_env():
    env = gym.make(config["env_name"], render_mode="rgb_array", hardcore=False) # Try Hardcore true, you will get the results that are shown in the GIF :/
    env = Monitor(env)
    return env


## **Part B: Define the environment**

In [ ]:
env_name = "BipedalWalker-v3"
config = {
    "policy_type": "MlpPolicy", # ActorCriticPolicy
    "total_timesteps": 150000,
    "env_name": env_name
}

## **Part C: Initialize the Real Time Data Logger - W&B**

A tool for visualizing and tracking your machine learning experiments.

In [ ]:
run = wandb.init(
    project="RL_163A_263A",
    config=config,
    sync_tensorboard=True,
    monitor_gym=True,
    save_code=False,
)

/usr/local/lib/python3.12/dist-packages/wandb/analytics/sentry.py:263: DeprecationWarning: The `Scope.user` setter is deprecated in favor of `Scope.set_user()`.
  self.scope.user = {"email": email}


## **Part D: Initialize the Video Recorder**
Every 20000 steps, we will record the next 200 steps of the experiment. This snapshots during the training allows to understand how the learning process evolves and it is one strategy to debug our code.


In [ ]:
env = DummyVecEnv([make_env])

class CustomVecVideoRecorder(VecVideoRecorder):
    def _start_recording(self) -> None:
        """Start a new recording with updated video name based on current step."""
        if self.recording:
            self._stop_recording()

        # Update video name with current step
        self.video_name = f"{self.name_prefix}-step-{self.step_id}-to-step-{self.step_id + self.video_length}.mp4"
        self.video_path = os.path.join(self.video_folder, self.video_name)

        self.recording = True

# Create video recording wrapper using our custom class
env = DummyVecEnv([make_env])
env = CustomVecVideoRecorder(
    env,
    video_folder=f"videos/{run.id}",
    record_video_trigger=lambda step: step % 20000 == 0,  # Record every 20000 steps
    video_length=200,  # Record 200 frames per video
    name_prefix=f"bipedal_training"
)

## **Part E: Define the RL algorithm to solve the control problem**
I strongly encourage you to review the following script to gain a deeper understanding of the experiment's setup, including hyperparameters, learning rate, FPS, epochs, and other details. You can find the script in the repository here: [Bipedal_Walker.py](https://github.com/Farama-Foundation/Gymnasium/blob/main/gymnasium/envs/box2d/bipedal_walker.py#L170)

In [ ]:
model = PPO(config["policy_type"], env, verbose=0, tensorboard_log=f"runs/{run.id}")

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarni

![schematic.png](https://pylessons.com/media/Tutorials/Reinforcement-learning-tutorial/BipedalWalker-v3-PPO/1.png)

### **Action Space**
These are actions that we are command to move our bipedal robot!

![Action_Space.JPG](https://pylessons.com/media/Tutorials/Reinforcement-learning-tutorial/BipedalWalker-v3-PPO/3.png)

In [ ]:
from pprint import pprint
pprint(model.action_space) # Torques

Box(-1.0, 1.0, (4,), float32)


### **Observation Space**

![Observation_Space.JPG](https://pylessons.com/media/Tutorials/Reinforcement-learning-tutorial/BipedalWalker-v3-PPO/2.png)

In [ ]:
pprint(model.observation_space) # Inputs to the Policy

Box([-3.1415927 -5.        -5.        -5.        -3.1415927 -5.
 -3.1415927 -5.        -0.        -3.1415927 -5.        -3.1415927
 -5.        -0.        -1.        -1.        -1.        -1.
 -1.        -1.        -1.        -1.        -1.        -1.       ], [3.1415927 5.        5.        5.        3.1415927 5.        3.1415927
 5.        5.        3.1415927 5.        3.1415927 5.        5.
 1.        1.        1.        1.        1.        1.        1.
 1.        1.        1.       ], (24,), float32)


### **Rewards**
Check line 131 and 595 in [Bipedal_Walker.py](https://github.com/Farama-Foundation/Gymnasium/blob/main/gymnasium/envs/box2d/bipedal_walker.py#L170) !

## **Part F: Training**

### **Neural Network**

Our implementation uses a Multilayer Perceptron (MLP) as the foundation for both the actor and critic networks. MLPs are particularly well-suited for continuous control tasks like BipedalWalker because:

1. Input Processing: They can effectively process the continuous state observations (joint angles, velocities, etc.). A neural network can approximate any continuous function,
2. Non-linear Mappings: Through multiple layers with non-linear activations, MLPs can learn complex relationships between states and actions
3. Continuous Outputs: They can output continuous action values with appropriate activation functions

The mathematical foundation of each layer works as follows:

For a hidden layer l, the output $h^{(l)}$ is computed as:

$h^{(l)} = \sigma(W^{(l)}h^{(l-1)} + b^{(l)})$, where $W^{(l)}$

Where:
- $W^(l)$ is the weight matrix for layer l
- $b^(l)$ is the bias vector
- $h^{(l-1)}$ is the input from the previous layer
- $\sigma$ is the activation function (typically ReLU for hidden layers)

This architecture enables our agent to learn:
- State-dependent features through early layers
- Action-value mappings through middle layers
- Final policy and value outputs through the output layers




![This_photo.jpg](https://www.researchgate.net/publication/365129352/figure/fig1/AS:11431281113458572@1673924170879/Basic-artificial-neural-network-structure-It-consists-of-N-inputs-ranging-from.png)




In [ ]:
pprint(model.policy)

ActorCriticPolicy(
  (features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (pi_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (vf_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (mlp_extractor): MlpExtractor(
    (policy_net): Sequential(
      (0): Linear(in_features=24, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): Tanh()
    )
    (value_net): Sequential(
      (0): Linear(in_features=24, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): Tanh()
    )
  )
  (action_net): Linear(in_features=64, out_features=4, bias=True)
  (value_net): Linear(in_features=64, out_features=1, bias=True)
)


### **Actor and Critic**
*   *The Actor network (policy network) π(a|s)* **learns a policy by directly mapping states to actions**, outputting either action probabilities for discrete action spaces or continuous action values for continuous action spaces. It gets updated using the advantage function computed from the Critic's feedback to improve its action selection strategy.
*  *The Critic network (value network)* estimates either the state value V(s) or state-action value Q(s,a), serving as a learned baseline that helps reduce variance in policy updates. By evaluating the Actor's decisions through value estimation,**it provides a more reliable learning signal than raw returns**, enabling stable policy improvement while reducing the variance in gradient estimates.


![framework.png](https://pylessons.com/media/Tutorials/Reinforcement-learning-tutorial/BipedalWalker-v3-PPO/5.png)

In [ ]:
model.learn(
    total_timesteps=config["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}", # Save the Intermediate Policies. (.pt)
        verbose=0
    ),
)

PPO_path = os.path.join('Training', 'Saved Models', 'PPO_BipedalWalker_150k')
model.save(PPO_path)

evaluate_policy(model, env, n_eval_episodes=10, render=True)
run.finish()

Saving video to /content/videos/9jzpz233/bipedal_training-step-0-to-step-200.mp4


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())


Moviepy - Building video /content/videos/9jzpz233/bipedal_training-step-0-to-step-200.mp4.
Moviepy - Writing video /content/videos/9jzpz233/bipedal_training-step-0-to-step-200.mp4



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Moviepy - Done !
Moviepy - video ready /content/videos/9jzpz233/bipedal_training-step-0-to-step-200.mp4
Saving video to /content/videos/9jzpz233/bipedal_training-step-20000-to-step-20200.mp4
Moviepy - Building video /content/videos/9jzpz233/bipedal_training-step-20000-to-step-20200.mp4.
Moviepy - Writing video /content/videos/9jzpz233/bipedal_training-step-20000-to-step-20200.mp4



...

Moviepy - Done !
Moviepy - video ready /content/videos/9jzpz233/bipedal_training-step-140000-to-step-140200.mp4


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/save_util.py:284: UserWarning: Path 'Training/Saved Models' does not exist. Will create it.
  warnings.warn(f"Path '{path.parent}' does not exist. Will create it.")


Saving video to /content/videos/z5ll76j1/bipedal_training-step-0-to-step-200.mp4
Moviepy - Building video /content/videos/z5ll76j1/bipedal_training-step-0-to-step-200.mp4.
Moviepy - Writing video /content/videos/z5ll76j1/bipedal_training-step-0-to-step-200.mp4



Moviepy - Done !
Moviepy - video ready /content/videos/z5ll76j1/bipedal_training-step-0-to-step-200.mp4


global_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇████
rollout/ep_len_mean,▁▃▃▃▄▄▅▅▅▆▆▆▆▇▇▇▇▇▇████████▇▇▇▇▇▇▇▇▇▇▇▆▆
rollout/ep_rew_mean,▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▇▇▇█
time/fps,█▁▁▄▄▅▅▆▇▇▇▇▇▇▇▆▆▇▇▇███████▇▇▇▇▇▇▇▇▇▇▇▇▇
train/approx_kl,▆▃▄▅▄▄▄▃▄▂▅▄▅▂▆▆▆▆▅▃▆▁▁▃▇▅▇▆▅▅▁▁▂▇▄▄▇▅▃█
train/clip_fraction,▃▂▄▄▁▅▄▄▄▄▄▅▆▂▃▃▇▆▅▆▂▇▁▁▃▅▇▆▁▃▄▃█▇▅█▄▄▂▇
train/clip_range,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/entropy_loss,▁▁▁▁▃▃▃▃▃▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇█████
train/explained_variance,▄█▆▅▁▆▆▅▄▆█▅▇▆▇████▆▇▇▇▅▇█▇█▇█▅▇▅▇█▇████
train/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+4,...


By tweaking the parameters in the repository, you could obtain different walking strategies.

![Walking_Strategies.png](https://pylessons.com/media/Tutorials/Reinforcement-learning-tutorial/BipedalWalker-v3-PPO/4.png)

In [ ]:
# Command to delete the folder where the videos are stored. Only used it if you don't have more space after many training runs
#!rm -rf videos/ training_records/ wandb/

References:
*    BipedalWalker-v3 with Continuous Proximal Policy Optimization ([link](https://pylessons.com/BipedalWalker-v3-PPO))
*   Bipedal Walker ([link](https://www.gymlibrary.dev/environments/box2d/bipedal_walker/))
* If you are new to PyTorch, you can check out this notebook ([link](https://colab.research.google.com/github/omarsar/pytorch_notebooks/blob/master/pytorch_quick_start.ipynb)) to have a quick start!
  
Courses:
* Stanford Deep Reinforcement Learning ([link](https://cs224r.stanford.edu/))
* Berkeley Deep RL ([link](https://rail.eecs.berkeley.edu/deeprlcourse/))



